In [ ]:
"""
=============================================================================
 ÉTUDE 4 — GATING PAR LA CONFIANCE ET CALIBRATION DES CLASSIFIEURS
=============================================================================
 Le mécanisme de Confidence-Gated Fusion consiste à neutraliser la
 contribution d'un module lorsque sa confiance est insuffisante :

     s(τ) = s × 1[max(softmax) ≥ τ]

 L'étude évalue ce mécanisme sur les deux classifieurs du pipeline (M3 et
 M4) et examine la condition qui le rend opérant : la confiance doit être
 informative, c'est-à-dire plus élevée sur les prédictions correctes que
 sur les prédictions erronées. Un classifieur sur-confiant rend le gating
 inopérant, puisque presque aucune image ne franchit le seuil par le bas.

 Entrée : resultats_phase4_v5.csv
 GPU    : non nécessaire
=============================================================================
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
CSV_PATH = '/kaggle/input/datasets/abdallahouahmane/resultas-phase4/resultats_phase4_v5.csv'

W = {'ctr': 0.15, 'cong': 0.40, 'calcif': 0.10, 'cardio': 0.35}
COLONNE = {'ctr': 's_ctr', 'cong': 's_cong',
           'calcif': 's_calcif', 'cardio': 'prob_cardio'}

TAUS = np.arange(0.20, 1.00, 0.05)
TOLERANCE = 0.005
N_BINS = 10                # bacs de calibration

print("=" * 62)
print("  ÉTUDE 4 — GATING PAR LA CONFIANCE")
print("=" * 62)

df = pd.read_csv(CSV_PATH)
df['y_true'] = df['label_reel'].apply(
    lambda x: 1 if ('Cardiomegaly' in str(x) or 'Edema' in str(x)) else 0)
y = df['y_true'].values

print(f"\n  CSV chargé : {len(df)} images")
print(f"  Positifs : {y.sum()} | Négatifs : {(y == 0).sum()}")

# Confiance de M4 : maximum du softmax sur les trois classes
df['confiance_m4'] = df[['prob_normal', 'prob_cardio',
                         'prob_edema']].max(axis=1)


def score_v5(s_calcif=None, s_cardio=None):
    """Formule V5, dont les composantes M3 et M4 sont substituables."""
    c = df['s_calcif'] if s_calcif is None else s_calcif
    d = df['prob_cardio'] if s_cardio is None else s_cardio
    return (W['ctr'] * df['s_ctr'] + W['cong'] * df['s_cong']
            + W['calcif'] * c + W['cardio'] * d)


auc_v5 = roc_auc_score(y, score_v5())
print(f"  AUC de référence (V5, sans gating) : {auc_v5:.4f}")

# ============================================================
# 1. DISTRIBUTION DE LA CONFIANCE
# ============================================================
print(f"\n{'='*62}")
print("  1. DISTRIBUTION DE LA CONFIANCE DES DEUX CLASSIFIEURS")
print(f"{'='*62}\n")

print(f"  {'':16} {'M3 (calcique)':>16} {'M4 (risque)':>16}")
print("  " + "-" * 50)
for libelle, fonction in [('minimum', 'min'), ('médiane', 'median'),
                          ('moyenne', 'mean'), ('maximum', 'max'),
                          ('écart-type', 'std')]:
    v3 = getattr(df['confiance_m3'], fonction)()
    v4 = getattr(df['confiance_m4'], fonction)()
    print(f"  {libelle:<16} {v3:>16.4f} {v4:>16.4f}")

print(f"\n  Proportion d'images au-dessus de chaque seuil :")
print(f"  {'seuil':>8} {'M3 actif':>12} {'M4 actif':>12}")
print("  " + "-" * 34)
for tau in [0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]:
    print(f"  {tau:>8.2f} {(df['confiance_m3'] >= tau).mean()*100:>11.1f}% "
          f"{(df['confiance_m4'] >= tau).mean()*100:>11.1f}%")

pct_m4_090 = (df['confiance_m4'] >= 0.90).mean() * 100
print(f"\n  {pct_m4_090:.1f} % des images reçoivent de M4 une confiance ≥ 0.90.")
print(f"  Un seuil de gating placé sous cette valeur ne désactiverait")
print(f"  presque aucune prédiction : le mécanisme serait sans effet.")

# ============================================================
# 2. LA CONFIANCE EST-ELLE INFORMATIVE ?
# ============================================================
print(f"\n{'='*62}")
print("  2. LA CONFIANCE DISCRIMINE-T-ELLE LES ERREURS ?")
print(f"{'='*62}\n")

# M4 : la prédiction est-elle cohérente avec l'étiquette binaire ?
pred_m4 = (1 - df['prob_normal'] >= 0.5).astype(int)
correct_m4 = (pred_m4 == y)

print(f"  Module 4 — confiance moyenne selon l'issue de la prédiction :")
print(f"    prédictions correctes ({correct_m4.sum():>3}) : "
      f"{df.loc[correct_m4, 'confiance_m4'].mean():.4f}")
print(f"    prédictions erronées  ({(~correct_m4).sum():>3}) : "
      f"{df.loc[~correct_m4, 'confiance_m4'].mean():.4f}")
ecart_m4 = (df.loc[correct_m4, 'confiance_m4'].mean()
            - df.loc[~correct_m4, 'confiance_m4'].mean())
print(f"    écart : {ecart_m4:+.4f}")

auc_conf_m4 = roc_auc_score(correct_m4.astype(int), df['confiance_m4'])
print(f"\n  Capacité de la confiance de M4 à prédire sa propre justesse :")
print(f"    AUC = {auc_conf_m4:.4f}")
if auc_conf_m4 < 0.60:
    print(f"    Proche du hasard : la confiance n'identifie pas les erreurs.")
else:
    print(f"    La confiance porte une information sur la justesse.")

# Erreur de calibration attendue (ECE) pour M4
bornes = np.linspace(0, 1, N_BINS + 1)
ece, details = 0.0, []
for i in range(N_BINS):
    m = (df['confiance_m4'] > bornes[i]) & (df['confiance_m4'] <= bornes[i + 1])
    if m.sum() > 0:
        conf_moy = df.loc[m, 'confiance_m4'].mean()
        just_moy = correct_m4[m].mean()
        ece += (m.sum() / len(df)) * abs(conf_moy - just_moy)
        details.append((bornes[i], bornes[i + 1], int(m.sum()),
                        conf_moy, just_moy))

print(f"\n  Erreur de calibration attendue (ECE) de M4 : {ece:.4f}")
print(f"\n  {'intervalle':>16} {'n':>6} {'confiance':>11} {'justesse':>10} "
      f"{'écart':>8}")
print("  " + "-" * 56)
for lo, hi, n, conf, just in details:
    print(f"  [{lo:.2f} ; {hi:.2f}]{'':>4} {n:>6} {conf:>11.4f} "
          f"{just:>10.4f} {conf - just:>+8.4f}")

# ============================================================
# 3. GATING DE M3
# ============================================================
print(f"\n{'='*62}")
print("  3. GATING DE LA COMPOSANTE CALCIQUE (M3)")
print(f"{'='*62}\n")
print(f"  s_calcif(τ) = s_calcif × 1[confiance_M3 ≥ τ]\n")
print(f"  {'τ':>7} {'M3 actif':>11} {'AUC':>10} {'Δ vs V5':>10}")
print("  " + "-" * 42)

gating_m3 = {}
for tau in TAUS:
    tau = round(tau, 2)
    porte = (df['confiance_m3'] >= tau).astype(float)
    s = score_v5(s_calcif=df['s_calcif'] * porte)
    a = roc_auc_score(y, s)
    gating_m3[tau] = {'auc': a, 'delta': a - auc_v5,
                      'actif': porte.mean() * 100, 'scores': s}
    print(f"  {tau:>7.2f} {porte.mean()*100:>10.1f}% {a:>10.4f} "
          f"{a - auc_v5:>+10.4f}")

tau_m3 = max(gating_m3, key=lambda t: gating_m3[t]['auc'])
print(f"\n  Meilleur seuil : τ = {tau_m3:.2f}  "
      f"(AUC = {gating_m3[tau_m3]['auc']:.4f}, "
      f"gain {gating_m3[tau_m3]['delta']:+.4f})")

# ============================================================
# 4. GATING DE M4
# ============================================================
print(f"\n{'='*62}")
print("  4. GATING DE LA COMPOSANTE CARDIOMÉGALIE (M4)")
print(f"{'='*62}\n")
print(f"  prob_cardio(τ) = prob_cardio × 1[confiance_M4 ≥ τ]\n")
print(f"  {'τ':>7} {'M4 actif':>11} {'AUC':>10} {'Δ vs V5':>10}")
print("  " + "-" * 42)

gating_m4 = {}
for tau in TAUS:
    tau = round(tau, 2)
    porte = (df['confiance_m4'] >= tau).astype(float)
    s = score_v5(s_cardio=df['prob_cardio'] * porte)
    a = roc_auc_score(y, s)
    gating_m4[tau] = {'auc': a, 'delta': a - auc_v5,
                      'actif': porte.mean() * 100, 'scores': s}
    print(f"  {tau:>7.2f} {porte.mean()*100:>10.1f}% {a:>10.4f} "
          f"{a - auc_v5:>+10.4f}")

tau_m4 = max(gating_m4, key=lambda t: gating_m4[t]['auc'])
print(f"\n  Meilleur seuil : τ = {tau_m4:.2f}  "
      f"(AUC = {gating_m4[tau_m4]['auc']:.4f}, "
      f"gain {gating_m4[tau_m4]['delta']:+.4f})")

# ============================================================
# 5. CONFIANCE PAR GROUPE CLINIQUE
# ============================================================
print(f"\n{'='*62}")
print("  5. CONFIANCE PAR GROUPE CLINIQUE")
print(f"{'='*62}\n")

groupes = {
    'Normal':        df[df['label_reel'] == 'No Finding'],
    'Cardiomégalie': df[df['label_reel'].str.contains('Cardiomegaly', na=False)],
    'Œdème':         df[df['label_reel'].str.contains('Edema', na=False)],
}
print(f"  {'Groupe':<16} {'conf. M3':>10} {'conf. M4':>10} "
      f"{'M4 ≥ 0.90':>11}")
print("  " + "-" * 50)
for nom, sub in groupes.items():
    if len(sub):
        print(f"  {nom:<16} {sub['confiance_m3'].mean():>10.4f} "
              f"{sub['confiance_m4'].mean():>10.4f} "
              f"{(sub['confiance_m4'] >= 0.90).mean()*100:>10.1f}%")

# ============================================================
# 6. VISUALISATION
# ============================================================
fig = plt.figure(figsize=(21, 13))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.30)

# ── Panel 1 : effet du gating ─────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
taus = sorted(gating_m3)
ax1.plot(taus, [gating_m3[t]['auc'] for t in taus], '-o', ms=5, lw=2.2,
         color='#B45309', label='Gating de M3 (calcique)')
ax1.plot(taus, [gating_m4[t]['auc'] for t in taus], '-s', ms=5, lw=2.2,
         color='#8E44AD', label='Gating de M4 (cardiomégalie)')
ax1.axhline(auc_v5, color='#C0392B', ls='--', lw=2,
            label=f'V5 sans gating ({auc_v5:.4f})')
ax1.fill_between(taus, auc_v5 - TOLERANCE, auc_v5 + TOLERANCE,
                 alpha=0.12, color='gray', label=f'Tolérance ±{TOLERANCE}')
ax1.set_xlabel('Seuil de confiance τ', fontsize=11)
ax1.set_ylabel('AUC-ROC', fontsize=11)
ax1.set_title("Étude 4 — Effet du gating sur l'AUC du pipeline\n"
              's(τ) = s × 1[confiance ≥ τ]', fontweight='bold', fontsize=11)
ax1.legend(fontsize=9, loc='lower left')
ax1.grid(alpha=0.3)

# ── Panel 2 : taux d'activation ───────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
ax2.plot(taus, [gating_m3[t]['actif'] for t in taus], '-o', ms=4, lw=2.2,
         color='#B45309', label='M3')
ax2.plot(taus, [gating_m4[t]['actif'] for t in taus], '-s', ms=4, lw=2.2,
         color='#8E44AD', label='M4')
ax2.axhline(50, color='gray', ls=':', lw=1.2)
ax2.set_xlabel('Seuil τ')
ax2.set_ylabel('% de modules actifs')
ax2.set_title("Taux d'activation\nselon le seuil", fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

# ── Panel 3 : distributions de confiance ──────────────────
ax3 = fig.add_subplot(gs[1, 0])
ax3.hist(df['confiance_m3'], bins=30, alpha=0.62, color='#B45309',
         label='M3 (calcique)', edgecolor='white')
ax3.hist(df['confiance_m4'], bins=30, alpha=0.62, color='#8E44AD',
         label='M4 (risque)', edgecolor='white')
ax3.axvline(0.90, color='#C0392B', ls='--', lw=1.8, label='0.90')
ax3.set_xlabel('Confiance (max du softmax)')
ax3.set_ylabel('Effectif')
ax3.set_title('Distribution de la confiance\ndes deux classifieurs',
              fontweight='bold')
ax3.legend(fontsize=8)
ax3.grid(alpha=0.3, axis='y')

# ── Panel 4 : diagramme de fiabilité de M4 ────────────────
ax4 = fig.add_subplot(gs[1, 1])
if details:
    centres = [(lo + hi) / 2 for lo, hi, _, _, _ in details]
    justesses = [j for _, _, _, _, j in details]
    effectifs = [n for _, _, n, _, _ in details]
    ax4.bar(centres, justesses, width=0.09, alpha=0.8, color='#8E44AD',
            edgecolor='white', label='Justesse observée')
    ax4.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Calibration parfaite')
    for c, j, n in zip(centres, justesses, effectifs):
        ax4.text(c, j + 0.02, f'n={n}', ha='center', fontsize=7)
ax4.set_xlim(0, 1)
ax4.set_ylim(0, 1.08)
ax4.set_xlabel('Confiance annoncée')
ax4.set_ylabel('Justesse observée')
ax4.set_title(f'Diagramme de fiabilité de M4\nECE = {ece:.4f}',
              fontweight='bold')
ax4.legend(fontsize=8, loc='upper left')
ax4.grid(alpha=0.3)

# ── Panel 5 : confiance selon la justesse ─────────────────
ax5 = fig.add_subplot(gs[1, 2])
bp = ax5.boxplot([df.loc[correct_m4, 'confiance_m4'].values,
                  df.loc[~correct_m4, 'confiance_m4'].values],
                 tick_labels=['Prédiction\ncorrecte', 'Prédiction\nerronée'],
                 patch_artist=True, notch=True)
for patch, c in zip(bp['boxes'], ['#16A34A', '#C0392B']):
    patch.set_facecolor(c)
    patch.set_alpha(0.72)
ax5.set_ylabel('Confiance de M4')
ax5.set_title(f'La confiance sépare-t-elle\nles erreurs ? '
              f'(AUC = {auc_conf_m4:.3f})', fontweight='bold')
ax5.grid(alpha=0.3, axis='y')

plt.suptitle(
    f'ÉTUDE 4 — Gating par la confiance | V5 : AUC = {auc_v5:.4f} | '
    f'gain maximal M3 {gating_m3[tau_m3]["delta"]:+.4f}, '
    f'M4 {gating_m4[tau_m4]["delta"]:+.4f} | n = {len(df)}',
    fontsize=13, fontweight='bold')

for ext in ('png', 'svg'):
    plt.savefig(f'etude4_gating_confiance.{ext}', dpi=300,
                bbox_inches='tight', facecolor='white')
plt.show()

# ============================================================
# 7. SYNTHÈSE
# ============================================================
print(f"\n{'='*62}")
print("  SYNTHÈSE — ÉTUDE 4")
print(f"{'='*62}\n")

gain_m3 = gating_m3[tau_m3]['delta']
gain_m4 = gating_m4[tau_m4]['delta']

print(f"  Gain maximal du gating de M3 : {gain_m3:+.4f} (τ = {tau_m3:.2f})")
print(f"  Gain maximal du gating de M4 : {gain_m4:+.4f} (τ = {tau_m4:.2f})")

if max(gain_m3, gain_m4) <= TOLERANCE:
    print(f"\n  Aucun des deux gatings n'atteint le seuil de significativité")
    print(f"  de {TOLERANCE}. Le mécanisme n'est pas retenu dans la formule.")

print(f"\n  Deux raisons expliquent ce résultat :")
print(f"\n  La confiance de M4 est mal calibrée. Son ECE atteint {ece:.4f} et")
print(f"  {pct_m4_090:.1f} % des images franchissent le seuil de 0.90, si bien")
print(f"  qu'aucun seuil réaliste ne désactive un nombre utile de prédictions.")
print(f"\n  La confiance ne sépare pas les erreurs. Son pouvoir de prédire")
print(f"  la justesse du modèle atteint une AUC de {auc_conf_m4:.4f} ; l'écart")
print(f"  de confiance entre prédictions correctes et erronées est de")
print(f"  {ecart_m4:+.4f}. Neutraliser les prédictions peu confiantes revient")
print(f"  donc à écarter des cas majoritairement corrects.")
print(f"\n  Ce résultat négatif documente une limite de calibration et")
print(f"  motive la perspective d'un recalibrage (température, Platt).")

print(f"\n  Fichiers générés : etude4_gating_confiance.png / .svg")